# Set 8.1: Train a classification CNN and save it with its passport

**How to use this notebook**

1. In Colab: *Runtime → Change runtime type → T4 GPU*.
2. *Runtime → Run all*. The small CNN trains in about 15 minutes, InceptionV3 in about 10.
3. At the end, the browser downloads `set8_models.zip`. Unzip it into the app's `models/` folder.
4. In the ForensicsApp, open an image and click **Deep learning → CNN classification** (Set 8.2).

Each model is saved as three files:

| File | What it is |
|---|---|
| `<name>.keras` | the trained weights (Keras format, for Colab / TensorFlow) |
| `<name>.onnx` | the same network as ONNX, which the app runs with OpenCV (no TensorFlow needed) |
| `<name>_passport.json` | the **model passport**: dataset and sample sizes, class names, preprocessing, training history, test accuracy |

The InceptionV3 transfer-learning part at the end is optional.

In [ ]:
# ONNX export needs onnx and tf2onnx; --no-deps keeps Colab's TensorFlow and protobuf untouched
!pip install -q onnx onnxruntime
!pip install -q --no-deps tf2onnx

In [ ]:
import datetime
import json
import zipfile

import cv2
import keras
import numpy as np
import tensorflow as tf
from keras import layers
from matplotlib import pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

keras.utils.set_random_seed(0)
print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

CLASSES = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE
SAVED = []  # files to download at the end

## Dataset: CIFAR-10

60 000 colour images of 32 × 32 px in 10 classes. The last 5 000 training images are kept for validation.
The images stay `uint8` here; the preprocessing (below) is done in the input pipeline and written into the passport,
so the app can repeat it exactly.

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
y_train, y_test = y_train.ravel(), y_test.ravel()
VALIDATION = 5000
x_val, y_val = x_train[-VALIDATION:], y_train[-VALIDATION:]
x_train, y_train = x_train[:-VALIDATION], y_train[:-VALIDATION]
print("train", x_train.shape, "validation", x_val.shape, "test", x_test.shape)

fig, axes = plt.subplots(10, 10, figsize=(12, 12))
for row, name in enumerate(CLASSES):
    for col, index in enumerate(np.where(y_train == row)[0][:10]):
        axes[row, col].imshow(x_train[index])
        axes[row, col].axis("off")
    axes[row, 0].set_title(name, loc="left", fontsize=9)
plt.show()

## Preprocessing

`input = (pixel / 255 − mean) / std`, after resizing to the model's input size. For the small CNN mean = 0 and
std = 1, so pixels are just scaled to [0, 1]. Training images are also randomly flipped and shifted (augmentation);
that is for training only and is not part of the preprocessing.

In [ ]:
augment = keras.Sequential([layers.RandomFlip("horizontal"), layers.RandomTranslation(0.1, 0.1)])


def dataset(images, labels, size=(32, 32), mean=0.0, std=1.0, train=False):
    # uint8 images -> batches of preprocessed float32 images, as the model (and the app) expects them
    ds = tf.data.Dataset.from_tensor_slices((images, labels))
    if train:
        ds = ds.shuffle(10_000, seed=0)

    def prepare(x, y):
        x = tf.cast(x, tf.float32)
        if train:
            x = augment(x, training=True)
        if tuple(size) != (32, 32):
            x = tf.image.resize(x, size, method="bilinear")
        return (x / 255.0 - mean) / std, y

    return ds.batch(BATCH_SIZE).map(prepare, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

## Save weights + passport

`save_with_passport` saves the `.keras` weights, exports the `.onnx` file, writes the passport and finally checks
that OpenCV (what the app uses) gives the same predictions as Keras.

In [ ]:
def layer_rows(model):
    rows = []
    for layer in model.layers:
        try:
            shape = str(tuple(layer.output.shape))
        except Exception:
            shape = "?"
        rows.append(f"{layer.name} ({layer.__class__.__name__}) -> {shape}")
    return rows


def export_onnx(model, path, size):
    try:
        model.export(path, format="onnx")
    except Exception as error:  # older Keras: convert a traced function with tf2onnx directly
        print("model.export failed, using tf2onnx:", error)
        import tf2onnx
        spec = (tf.TensorSpec((None, *size, 3), tf.float32, name="image"),)
        function = tf.function(lambda x: model(x, training=False), input_signature=spec)
        tf2onnx.convert.from_function(function, input_signature=spec, opset=13, output_path=path)


def check_with_opencv(model, path, size, mean, std, count=32):
    x = next(iter(dataset(x_test[:count], y_test[:count], size, mean, std)))[0].numpy()
    expected = model.predict(x, verbose=0)
    net = cv2.dnn.readNetFromONNX(path)
    got = []
    for image in x:
        net.setInput(image[None])
        got.append(net.forward().reshape(-1))
    got = np.array(got)
    same = (got.argmax(1) == expected.argmax(1)).mean()
    print(f"OpenCV vs Keras on {count} test images: same class for {same:.0%}, "
          f"largest probability difference {np.abs(got - expected).max():.1e}")


def save_with_passport(model, history, name, title, size=(32, 32), mean=0.0, std=1.0, training=None,
                       train_count=None):
    test_loss, test_accuracy = model.evaluate(dataset(x_test, y_test, size, mean, std), verbose=0)
    print(f"Test accuracy {test_accuracy:.3f}, test loss {test_loss:.3f}")
    model.save(f"{name}.keras")
    export_onnx(model, f"{name}.onnx", size)
    to_list = lambda value: [float(value)] * 3 if np.isscalar(value) else [float(v) for v in value]
    passport = {
        "name": title,
        "created": datetime.datetime.now().isoformat(timespec="seconds"),
        "framework": f"TensorFlow {tf.__version__}, Keras {keras.__version__}",
        "files": {"weights": f"{name}.keras", "onnx": f"{name}.onnx"},
        "architecture": {"parameters": int(model.count_params()), "layers": layer_rows(model)},
        "dataset": {
            "name": "CIFAR-10",
            "source": "https://www.cs.toronto.edu/~kriz/cifar.html (keras.datasets.cifar10)",
            "image_size": [32, 32],
            "train": int(train_count or len(x_train)),
            "validation": int(len(x_val)),
            "test": int(len(x_test)),
        },
        "classes": CLASSES,
        "preprocessing": {
            "input_size": [int(size[0]), int(size[1])],  # height, width
            "color": "RGB",
            "layout": "NHWC",
            "resize": "bilinear",
            "scale": 1 / 255,
            "mean": to_list(mean),
            "std": to_list(std),
            "steps": [
                "Convert the image to RGB (3 channels, 8 bits each)",
                f"Resize to {size[1]} x {size[0]} px (bilinear)",
                "Divide the pixel values by 255 (range 0 to 1)",
                *([f"Subtract the mean {mean} and divide by the standard deviation {std} (range -1 to 1)"]
                  if (mean, std) != (0.0, 1.0) else []),
                f"Batch of shape (1, {size[0]}, {size[1]}, 3), float32; the output is the probability of each class",
            ],
        },
        "training": {**(training or {}),
                     "history": {key: [float(v) for v in values] for key, values in history.history.items()}},
        "evaluation": {"test_accuracy": float(test_accuracy), "test_loss": float(test_loss)},
    }
    with open(f"{name}_passport.json", "w") as file:
        json.dump(passport, file, indent=2)
    SAVED.extend([f"{name}.keras", f"{name}.onnx", f"{name}_passport.json"])
    print("Saved", f"{name}.keras,", f"{name}.onnx,", f"{name}_passport.json")
    try:
        check_with_opencv(model, f"{name}.onnx", size, mean, std)
    except Exception as error:
        print("OpenCV check failed (the app may not be able to run this model):", error)
    return passport


def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for axis, key in zip(axes, ("loss", "accuracy")):
        axis.plot(history.history[key], label="training")
        axis.plot(history.history["val_" + key], label="validation")
        axis.set(title=f"{title}: {key}", xlabel="epoch")
        axis.legend()
    plt.show()


def plot_confusion(model, size=(32, 32), mean=0.0, std=1.0):
    predictions = model.predict(dataset(x_test, y_test, size, mean, std), verbose=0).argmax(1)
    ConfusionMatrixDisplay.from_predictions(y_test, predictions, display_labels=CLASSES, xticks_rotation=45,
                                            cmap="Blues", colorbar=False)
    plt.gcf().set_size_inches(8, 8)
    plt.show()

## Small CNN (VGG-style, 3 blocks)

The notebook's deeper VGG model, with batch normalization and dropout added so it trains faster and overfits less
(88 % test accuracy after 100 epochs instead of about 65 %).

In [ ]:
EPOCHS = 100
LEARNING_RATE = 1e-3


def vgg_block(model, filters, dropout):
    for _ in range(2):
        model.add(layers.Conv2D(filters, 3, padding="same", use_bias=False, kernel_initializer="he_uniform"))
        model.add(layers.BatchNormalization())
        model.add(layers.Activation("relu"))
    model.add(layers.MaxPooling2D())
    model.add(layers.Dropout(dropout))


cnn = keras.Sequential([keras.Input((32, 32, 3), name="image")], name="cifar10_cnn")
for filters, dropout in ((32, 0.2), (64, 0.3), (128, 0.4)):
    vgg_block(cnn, filters, dropout)
cnn.add(layers.Flatten())
cnn.add(layers.Dense(128, activation="relu", kernel_initializer="he_uniform"))
cnn.add(layers.Dropout(0.5))
cnn.add(layers.Dense(len(CLASSES), activation="softmax"))
cnn.summary()

cnn.compile(optimizer=keras.optimizers.Adam(LEARNING_RATE), loss="sparse_categorical_crossentropy",
            metrics=["accuracy"])
cnn_history = cnn.fit(dataset(x_train, y_train, train=True), validation_data=dataset(x_val, y_val),
                      epochs=EPOCHS)

In [ ]:
plot_history(cnn_history, "Small CNN")
plot_confusion(cnn)
cnn_passport = save_with_passport(
    cnn, cnn_history, "cifar10_cnn", "CIFAR-10 CNN (VGG-style, 3 blocks)",
    training={"optimizer": "Adam", "learning_rate": LEARNING_RATE, "loss": "sparse categorical cross-entropy",
              "epochs": EPOCHS, "batch_size": BATCH_SIZE, "augmentation": "random horizontal flip, shift up to 10 %"},
)
print(json.dumps({key: value for key, value in cnn_passport.items() if key not in ("training", "architecture")},
                 indent=2))

## Optional: transfer learning with InceptionV3

ImageNet-pretrained InceptionV3 without its top, frozen; only a global average pooling, a 1024-unit dense layer and
the classifier are trained. InceptionV3 needs at least 75 × 75 px, so the images are resized to 96 × 96, and it
expects values in [−1, 1] (mean 0.5, std 0.5). About 73 % test accuracy after 20 epochs, about 10 minutes on a T4. The `.onnx` file is about 90 MB.

In [ ]:
TRANSFER_SIZE = (96, 96)
TRANSFER_EPOCHS = 20
keras.utils.set_random_seed(0)

base = keras.applications.InceptionV3(include_top=False, weights="imagenet", input_shape=(*TRANSFER_SIZE, 3))
base.trainable = False
inputs = keras.Input((*TRANSFER_SIZE, 3), name="image")
x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(1024, activation="relu")(x)
outputs = layers.Dense(len(CLASSES), activation="softmax")(x)
inception = keras.Model(inputs, outputs, name="cifar10_inceptionv3")

inception.compile(optimizer=keras.optimizers.Adam(LEARNING_RATE), loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
inception_history = inception.fit(dataset(x_train, y_train, TRANSFER_SIZE, 0.5, 0.5, train=True),
                                  validation_data=dataset(x_val, y_val, TRANSFER_SIZE, 0.5, 0.5),
                                  epochs=TRANSFER_EPOCHS)
plot_history(inception_history, "InceptionV3 (frozen)")
save_with_passport(
    inception, inception_history, "cifar10_inceptionv3", "CIFAR-10 InceptionV3 transfer learning (frozen base)",
    size=TRANSFER_SIZE, mean=0.5, std=0.5,
    training={"optimizer": "Adam", "learning_rate": LEARNING_RATE, "loss": "sparse categorical cross-entropy",
              "epochs": TRANSFER_EPOCHS, "batch_size": BATCH_SIZE,
              "augmentation": "random horizontal flip, shift up to 10 %",
              "base": "InceptionV3, ImageNet weights, frozen; trained: pooling, dense 1024, classifier"},
)

## Download

Unzip `set8_models.zip` into the app's `models/` folder.

In [ ]:
with zipfile.ZipFile("set8_models.zip", "w") as archive:
    for name in SAVED:
        archive.write(name)
print("Zipped:", ", ".join(SAVED))

from google.colab import files
files.download("set8_models.zip")